**Bitácora · Day 3 PrefixSpan (chestpain)**

- **v1** `Day3_prefixspan_v1` ← **this notebook**  
  PrefixSpan on group tags, visits kept; KMeans clusters + SPADE. *Files:* `patterns_chestpain.json`
- **v2** `Day3_prefixspan_v2`  
  Motifs per visit-pruned NMF component; psych-vs-troponin timing. *Files:* `nmf_k8_chestpain_v2.npz`
- **v3 hybrid** `Day3_prefixspan_v3_hybrid` (2026-10-06)  
  v2 + deterministic ties, held-out pass rate, same-day flag, Excel export. *Files:* `nmf_k8_chestpain_v2.npz`, `Day3_v3_hybrid_top3_patterns_table.xlsx`
- **v3 concept** `Day3_prefixspan_v3_concept` (2026-10-07)  
  Hybrid on concept-level tokens; k=7 bundles named directly (v2 matching only for other k). *Files:* `nmf_k8_chestpain_v3.npz`, `Day3_v3_concept_final_table_chestpain_k{K}.xlsx`, `token_map_chestpain_concept.csv`

# Day 3 · Sequential pattern mining — the day that answers dx_9487

**Today's goal:** produce a **named motif catalog** straight from the dx_9487 event
streams, with support counts, and directly test the anxiety/panic hypothesis. No
embeddings, no GPU, no licence. If you only have one day, spend it here.

### Summary

Sequential pattern mining searches for ordered subsequences that recur across patients. The
output is a list like `chest_pain -> anxiety -> ssri`, each with a count. That is exactly
the artifact RUW needs: discrete, nameable, countable, writable as a triage rule. It is
also the technique most ML curricula omit entirely, which is why it keeps getting skipped in
favour of much heavier machinery.

### Technique chosen per subtopic

| Subtopic | We use | Mentioned only |
|---|---|---|
| Itemset mining | *mentioned* | Apriori, FP-Growth, ECLAT |
| Sequence mining | **PrefixSpan**, implemented in ~25 lines | GSP, SPADE, CM-SPAM |
| Redundancy control | **closed patterns** | maximal patterns, top-k, MDL compression |
| Interest measure | **discriminative lift** | confidence, leverage, conviction |
| Interval logic | **signed gaps** | Allen's 13 relations, KarmaLego |
| Trajectory view | **transition graph** via networkx | process mining, `pm4py` |

We implement PrefixSpan by hand rather than installing SPMF. It is short, and understanding
the projection step is what lets you reason about why the output explodes.

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 150); pd.set_option('display.max_columns', 40)
import ethos_omop as eo

OUT_DIR = "../../inputs/chest_pain_cohort/"   # matches Day 1's extract_cohort() output dir
DOMAINS = ['Visit', 'Condition', 'Drug', 'Procedure', 'Measurement']

if not os.path.isdir(OUT_DIR):
    raise FileNotFoundError(
        f"{OUT_DIR}/ not found. Run Day 1's eo.extract_cohort(...) (or "
        f"eo.extract_omop(...) with OUT_DIR='omop_extract') first, or "
        f"python make_demo_extract.py for a synthetic stand-in.")

concept  = eo.load_concept(OUT_DIR)
codesets = eo.build_codesets(concept, verbose=False)
events   = eo.tag_events(eo.load_events(OUT_DIR, domains=DOMAINS), codesets)
person   = eo.load_person(OUT_DIR)
truth    = eo.load_truth(OUT_DIR)                   # None against real data

print(f"events {events.shape}   patients {events.person_id.nunique()}")
print(f"codeset coverage: {events.in_codeset.mean():.1%} of events carry a clinical tag")
print(f"truth labels: {'present (DEMO extract)' if truth is not None else 'ABSENT (real data)'}")
events.head(6)

In [ ]:
seqs = eo.sequences(events, col='tag', dedup_same_day=True, collapse_repeats=True)
pids = sorted(seqs)
db   = [seqs[p] for p in pids]

print("example timeline:")
print(' -> '.join(db[1][:12]))
print("\nmedian sequence length:", int(np.median([len(s) for s in db])))

### The two preprocessing decisions, restated

Both live in `eo.sequences()` and both are arbitrary:

1. **Same-day ties.** Claims carry a date, not a timestamp, so `DOMAIN_ORDER` breaks ties.
   Visit, condition, procedure, measurement, drug, observation.
2. **Refill collapsing.** Consecutive repeats of the same tag become one token, otherwise
   every pattern is `metformin -> metformin -> metformin`. Chronicity is real signal, so
   capture it as a separate feature — fill count, days supply — not as sequence position.

## 1 · PrefixSpan, implemented

In [ ]:
import collections

def prefixspan(db, min_sup, max_len=4):
    """Minimal PrefixSpan over single-item sequences.

    db      : list of lists of tokens
    min_sup : minimum number of sequences containing the pattern
    returns : DataFrame of (pattern, support, support_pct)
    """
    n, results = len(db), []

    def scan(projected, prefix):
        counts = collections.Counter()
        for seq in projected:
            for item in set(seq):                   # count each sequence once
                counts[item] += 1
        for item, c in counts.items():
            if c < min_sup:
                continue
            new_prefix = prefix + [item]
            results.append((tuple(new_prefix), c))
            if len(new_prefix) >= max_len:
                continue
            # project: keep the suffix after the FIRST occurrence of item
            nxt = [seq[seq.index(item) + 1:] for seq in projected if item in seq]
            if nxt:
                scan(nxt, new_prefix)

    scan([list(s) for s in db], [])
    return (pd.DataFrame(results, columns=['pattern', 'support'])
              .assign(support_pct=lambda d: (d.support / n * 100).round(1)))

In [ ]:
import hashlib, json

PAT_CACHE = "patterns_chestpain.json"   # PrefixSpan output + closed set; delete to force a re-mine
MIN_SUP   = int(0.04 * len(db))                     # 4% of patients

def db_key(db, min_sup, max_len):
    """Fingerprint of everything the mined patterns depend on."""
    h = hashlib.sha1()
    for s in db:
        h.update("\x1e".join(s).encode()); h.update(b"\x1f")
    return f"{len(db)}:{min_sup}:{max_len}:{h.hexdigest()}"

KEY    = db_key(db, MIN_SUP, 4)
cached = None
if os.path.exists(PAT_CACHE):
    with open(PAT_CACHE) as f:
        cached = json.load(f)
    if cached.get('key') != KEY:                    # stale: different sequences / support / max_len
        cached = None

if cached is not None:
    pats = pd.DataFrame({'pattern':     [tuple(p) for p, _, _ in cached['pats']],
                         'support':     [s for _, s, _ in cached['pats']],
                         'support_pct': [pc for _, _, pc in cached['pats']]})
    print(f"patterns loaded from {PAT_CACHE}")
else:
    pats = prefixspan(db, MIN_SUP, max_len=4)
    pats = (pats[pats.pattern.map(len) >= 3]
            .sort_values('support', ascending=False).reset_index(drop=True))
print(f"{len(pats)} patterns of length >= 3 at support {MIN_SUP}\n")
display(pats.head(12))


## 1b · SPADE, implemented

PrefixSpan (above) mines **horizontally**: it re-scans a shrinking projected database at
every recursion step. SPADE (Zaki, 2001) mines **vertically**: build one id-list per item up
front — `item -> {sequence_id: positions}` — then grow patterns by *joining* id-lists
instead of touching the raw sequences again.

The pruning is the **equivalence class**: all frequent patterns sharing a prefix `P` form the
class `[P]`. A child of `P -> a` is only ever built by joining it with its siblings in `[P]`,
because `P -> a -> b` can only be frequent if `P -> b` is (Apriori). Candidates outside the
class are never tried.

Same rule as PrefixSpan for what counts as "next": the extension item must occur after the
prefix's *first* occurrence in that sequence. Same rule in, same patterns and supports out —
only the data structure and the traversal differ. That equivalence is what the comparison
cell below checks.

In [ ]:
import bisect

def spade(db, min_sup, max_len=4):
    """SPADE: vertical id-lists + temporal joins inside prefix equivalence classes.

    Same contract as prefixspan() -- same input, same output schema -- so the two
    are directly comparable.

    A class [P] holds the frequent atoms P->a. Children of P->a are built only by
    joining P->a with its *siblings* in [P]. That is the Apriori prune: P->a->b can
    only be frequent if P->b is, so any b outside [P] is never even tried.
    """
    n, results = len(db), []

    # vertical id-lists: occ[item][seq_id] = sorted positions of item in that sequence
    occ = collections.defaultdict(lambda: collections.defaultdict(list))
    for sid, seq in enumerate(db):
        for pos, item in enumerate(seq):
            occ[item][sid].append(pos)

    def join(idlist, item):
        """Temporal join: {sid: end pos of prefix} x item -> {sid: first pos of item after it}."""
        item_occ, out = occ[item], {}
        for sid, last in idlist.items():
            idxs = item_occ.get(sid)
            if idxs:
                k = bisect.bisect_right(idxs, last)
                if k < len(idxs):
                    out[sid] = idxs[k]
        return out

    def enumerate_class(prefix, atoms):
        # atoms: [(item, idlist)] -- frequent one-item extensions of `prefix`
        for item, idl in atoms:
            new_prefix = prefix + [item]
            results.append((tuple(new_prefix), len(idl)))
            if len(new_prefix) >= max_len:
                continue
            child = []
            for sib, _ in atoms:                    # equivalence-class prune: siblings only
                j = join(idl, sib)
                if len(j) >= min_sup:
                    child.append((sib, j))
            if child:
                enumerate_class(new_prefix, child)

    # depth-1 class: frequent items, each with its first occurrence per sequence
    root = [(item, {sid: idxs[0] for sid, idxs in item_occ.items()})
            for item, item_occ in occ.items()]
    root = [(item, idl) for item, idl in root if len(idl) >= min_sup]
    enumerate_class([], root)

    return (pd.DataFrame(results, columns=['pattern', 'support'])
              .assign(support_pct=lambda d: (d.support / n * 100).round(1)))


In [ ]:
import time

t0 = time.perf_counter()
pats_ps = prefixspan(db, MIN_SUP, max_len=4)
t_ps = time.perf_counter() - t0

t0 = time.perf_counter()
pats_sp = spade(db, MIN_SUP, max_len=4)
t_sp = time.perf_counter() - t0

def norm(d):
    return d.set_index('pattern').support.sort_index()

same = norm(pats_ps).equals(norm(pats_sp))
print(f"PrefixSpan: {len(pats_ps)} patterns in {t_ps * 1000:.1f} ms")
print(f"SPADE:      {len(pats_sp)} patterns in {t_sp * 1000:.1f} ms")
print("identical pattern/support sets:", same)

In [ ]:
# Scaling test on synthetic data: double the number of sequences each step, time both miners.
# Zipf-distributed vocabulary (a few ambient codes, a long tail) like real claims; support is
# a fixed 4% so pattern count stays roughly constant and the curves isolate database size.
rng   = np.random.default_rng(0)
VOCAB = [f"c{i}" for i in range(60)]
w     = 1 / np.arange(1, len(VOCAB) + 1) ** 0.8; w /= w.sum()

def synth_db(n_seqs, length=30):
    return [list(rng.choice(VOCAB, size=length, p=w)) for _ in range(n_seqs)]

rows = []
for k in range(8, 14):                               # 256 ... 8192 sequences (~1 min total)
    n_seqs, sdb = 2 ** k, synth_db(2 ** k)
    ms = int(0.04 * n_seqs)
    t0 = time.perf_counter(); a = prefixspan(sdb, ms, max_len=4); t_a = time.perf_counter() - t0
    t0 = time.perf_counter(); b = spade(sdb, ms, max_len=4);      t_b = time.perf_counter() - t0
    assert norm(a).equals(norm(b)), f"miners disagree at n={n_seqs}"
    rows.append((n_seqs, len(a), t_a, t_b))
    print(f"n={n_seqs:>6}  patterns={len(a):>5}  prefixspan={t_a:7.2f}s  spade={t_b:7.2f}s  ratio={t_a / t_b:4.2f}")

bench = pd.DataFrame(rows, columns=['n_seqs', 'n_patterns', 'prefixspan_s', 'spade_s'])
ax = bench.plot(x='n_seqs', y=['prefixspan_s', 'spade_s'], marker='o', logx=True, logy=True, figsize=(6, 3.5))
ax.set_xlabel('sequences (doubling)'); ax.set_ylabel('seconds'); ax.set_title('PrefixSpan vs SPADE, synthetic'); plt.tight_layout(); plt.show()


## 2 · The redundancy problem, and closed patterns

Most of those are junk — prefixes of each other, or ambient primary care. **Closed** patterns
drop any pattern with a longer extension at the *same* support, removing prefix chains
without losing information.

Budget real time for this on the full vocabulary. Pattern pruning is the majority of the
work.

In [ ]:
def is_subseq(a, b):
    it = iter(b)
    return all(x in it for x in a)

rows = list(pats.itertuples(index=False))
if cached is not None and 'closed' in cached:
    closed_idx = cached['closed']
else:
    # only a pattern with the SAME support can dominate it, so compare within support buckets
    by_sup = collections.defaultdict(list)
    for q in rows:
        by_sup[q.support].append(q)
    closed_idx = [i for i, p in enumerate(rows)
                  if not any(len(q.pattern) > len(p.pattern) and is_subseq(p.pattern, q.pattern)
                             for q in by_sup[p.support])]
    with open(PAT_CACHE, 'w') as f:                 # persist for the next session
        json.dump({'key': KEY,
                   'pats': [[list(r.pattern), int(r.support), float(r.support_pct)] for r in rows],
                   'closed': closed_idx}, f)
    print(f"patterns + closed set saved to {PAT_CACHE}")
closed = [rows[i] for i in closed_idx]
print(f"{len(rows)} patterns -> {len(closed)} closed")


## 3 · Discriminative mining — where the motifs appear

Raw support finds whatever is common, which is mostly ambient care. What you want is
patterns common **inside** a group and rare **outside** it. That ratio is the interest
measure that produces usable motifs.

Group membership can come from anywhere — a weak label, a cluster from Day 2, or a codeset.
Here we use the Day 2 clustering, because that is what works when you have no labels.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

X   = eo.bag_of_codes(events, min_count=20).reindex(pids).fillna(0)
Z   = TruncatedSVD(min(24, X.shape[1] - 1), random_state=0).fit_transform(
          normalize(X.values.astype(float)))
clu = KMeans(7, n_init=10, random_state=0).fit(Z).labels_

def contains(seq, pat):
    it = iter(seq)
    return all(x in it for x in pat)

CAND = [tuple(p.pattern) for p in closed]

def discriminative(mask, candidates=CAND, sequences_db=None, min_in=0.12):
    sdb = sequences_db if sequences_db is not None else db
    out = []
    for p in candidates:
        h = np.array([contains(s, p) for s in sdb])
        si, so = h[mask].mean(), h[~mask].mean()
        if si >= min_in:
            out.append({'pattern': ' -> '.join(p), 'sup_in': round(si, 3),
                        'sup_out': round(so, 3), 'lift': round((si + .01) / (so + .01), 1)})
    if not out:
        return pd.DataFrame(columns=['pattern', 'sup_in', 'sup_out', 'lift'])
    return pd.DataFrame(out).sort_values('lift', ascending=False).reset_index(drop=True)

for c in range(7):
    d = discriminative(clu == c)
    if len(d):
        print(f"\n===== cluster {c}  (n={(clu == c).sum()})")
        display(d.head(3))

Each cluster now has **candidate motifs attached** — an ordered sequence, not just a set of
prevalent codes. That is the bridge from "cluster 2" to a name a clinician can argue with.

### The actual question: psych vs cardiac motifs

Clusters are one way to define groups when you have no labels. We also have a direct probe:
`chest_pain_labels()` from Day 1. Mine motifs discriminating `psych` from `cardiac`
directly — this is the hypothesis test, not a demo of the technique.

In [ ]:
weak = eo.chest_pain_labels(events)
wl   = weak.set_index('person_id').chest_pain_label

# All patients here have chest_pain by cohort construction, so no filtering needed --
# unlike a rare-indication drug, this IS the whole cohort. Still use LOCAL mining: a
# global 4% threshold is tuned for the full mixed cohort, not for either subgroup alone.
cp_pos = list(range(len(pids)))
y_cp   = np.array([wl.get(pids[i], 'unlabelled') for i in cp_pos])
print("chest_pain patients:", len(cp_pos),
      "  labelled:", int((y_cp != 'unlabelled').sum()))

pats_cp = prefixspan(db, max(3, int(0.08 * len(db))), max_len=4)
C_cp = [tuple(p) for p in pats_cp.pattern[pats_cp.pattern.map(len) >= 3]]

for g in ['psych', 'cardiac']:
    print(f"\n=== {g} evidence")
    display(discriminative(y_cp == g, C_cp, sequences_db=db, min_in=0.10).head(6))

**This is the deliverable.** Ordered patterns, with in-group and out-group support, that read
as sentences. No transformer, no embeddings, no non-commercial licence to negotiate.

## 4 · Interval relations — when order is not enough

Order says "A before B." It does not say "A the same week" versus "A two years before B,"
which is the actual dx_9487 discriminator: is the anxiety history old and unrelated, or did
a cardiac workup follow within days. Allen's interval algebra defines 13 relations; for
point events on dates, a **signed gap** carries most of the value.

In [ ]:
gaps = eo.signed_gap(events, 'chest_pain', 'troponin')
gaps = gaps.merge(weak, on='person_id', how='left')
gaps['chest_pain_label'] = gaps['chest_pain_label'].fillna('unlabelled')

plt.figure(figsize=(8.5, 3))
for name, g in gaps.groupby('chest_pain_label'):
    plt.scatter(g.days_a_to_b, np.random.normal(0, .05, len(g)),
                s=26, alpha=.75, label=f"{name} (n={len(g)})")
plt.axvline(0, c='k', lw=1); plt.yticks([])
plt.xlabel('days from chest_pain to troponin   (negative = troponin first)')
plt.legend(fontsize=8); plt.title('one signed integer, most of the workup-timing signal')
plt.tight_layout(); plt.show()

display(gaps.groupby('chest_pain_label').days_a_to_b
            .describe()[['count', 'mean', '50%']].round(0))

**One signed integer.** Worth remembering the next time the conversation turns to four weeks
of GPU pretraining.

Two cautions on real data. This feature is only meaningful where lookback covers the earlier
event — otherwise you are measuring enrolment, not biology. And for patients with no
troponin at all the feature is NULL, which is not the same as zero; carry an explicit
`no_workup_in_lookback` flag rather than imputing (this is exactly what
`cardiac_workup_outcome()` does formally in Day 5).

## 5 · Trajectory graph — the readable artifact

Aggregate consecutive transitions into a weighted directed graph. This is the closest thing
to a chest-pain triage flowchart, but derived from observed transitions rather than a
guideline document — Ethos has no chest-pain ruleset to start from, so the agreed method is
to start from the data and check clinical plausibility afterwards.

In [ ]:
import networkx as nx

FOCUS = {'chest_pain', 'ed_visit', 'ekg', 'troponin', 'stress_test',
         'coronary_angio', 'beta_blocker', 'statin', 'cad',
         'anxiety', 'panic_attack', 'ssri', 'benzodiazepine',
         'gerd', 'costochondritis'}

db_cp = [[c for c in seqs[p] if c in FOCUS] for p in pids if 'chest_pain' in seqs[p]]
print(f"{len(db_cp)} chest pain patients")

cnt = collections.Counter()
for s in db_cp:
    for u, v in zip(s, s[1:]):
        cnt[(u, v)] += 1
G = nx.DiGraph()
for (u, v), w in cnt.items():
    if w >= max(5, int(0.02 * len(db_cp))):
        G.add_edge(u, v, weight=w)
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

plt.figure(figsize=(10, 6.5))
pos = nx.spring_layout(G, seed=3, k=1.1)
w   = np.array([G[u][v]['weight'] for u, v in G.edges()], dtype=float)
nx.draw_networkx_nodes(G, pos, node_size=1600, node_color='#DCE6EF')
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edges(G, pos, width=1 + 4 * w / w.max(), alpha=.5,
                       arrowsize=12, connectionstyle='arc3,rad=0.08')
plt.title('chest pain trajectory graph -- observed transitions')
plt.axis('off'); plt.tight_layout(); plt.show()

Two arms should be visible out of `chest_pain`: one through troponin, stress test,
angiography and statins; the other through anxiety, panic_attack, ssri and benzodiazepine.
Whichever arm dominates numerically is the answer to the working hypothesis — let the graph
say which, rather than assuming.

## Exercises

1. Lower `MIN_SUP` to 1%. How many patterns, and is closed-pattern pruning enough?
2. Extend `prefixspan` to enforce a maximum time gap between consecutive items. Which
   patterns die? (This is the single highest-value extension for claims.)
3. Add `gerd` and `costochondritis` to the trajectory graph's `FOCUS` set with a lower edge
   threshold. Do either show up as a real third arm, or are they noise?
4. Re-run the whole notebook with `tag_events(..., fallback='drop')` so only curated
   codesets survive. Cleaner patterns, but what did you lose?
5. Write the top motif per group (psych, cardiac) as a named plain-English rule. **That
   list is the motif catalog** — the thing to actually put in front of RUW.

## Carry forward

A motif catalog for dx_9487 triage, obtained without a foundation model.